# **Data Generation — CRROS Project**

This notebook generates a synthetic, behavior-driven dataset simulating:
- Customer lifecycle
- Engagement (interactions)
- Purchases (transactions)

The goal is to create realistic data for downstream tasks:
- Churn prediction
- Purchase prediction
- Customer targeting

## Configuration

In [3]:
import numpy as np
import pandas as pd

In [4]:
# Reproducibility
np.random.seed(42)

# Date range
START_DATE = "2024-01-01"
END_DATE = "2026-03-31"

# Dataset sizes
N_CUSTOMERS = 8000
N_PRODUCTS = 120

# Convert to datetime
start_date = pd.to_datetime(START_DATE)
end_date = pd.to_datetime(END_DATE)

## Generate Customers

We simulate a diverse customer base with:
- Segmentation (high / medium / low value)
- Acquisition channels
- Demographics

These attributes will later influence:
- Interaction frequency
- Purchase behavior
- Churn patterns

In [5]:
def generate_customers(n_customers):
    
    customer_ids = [f"C{str(i).zfill(5)}" for i in range(1, n_customers + 1)]
    
    # Signup dates (biased slightly toward earlier dates)
    signup_dates = np.random.choice(
        pd.date_range(start_date, end_date - pd.Timedelta(days=90)),
        size=n_customers
    )
    
    # Segments
    segments = np.random.choice(
        ["high", "medium", "low"],
        size=n_customers,
        p=[0.2, 0.5, 0.3]
    )
    
    # Locations
    locations = np.random.choice(
        ["Delhi", "Mumbai", "Bangalore", "Kolkata", "Chennai",
         "Hyderabad", "Pune", "Bhubaneswar", "Ahmedabad", "Jaipur"],
        size=n_customers
    )
    
    # Acquisition channels
    channels = np.random.choice(
        ["organic", "ads", "referral"],
        size=n_customers,
        p=[0.4, 0.4, 0.2]
    )
    
    # Age (18–65)
    ages = np.random.normal(loc=35, scale=10, size=n_customers).astype(int)
    ages = np.clip(ages, 18, 65)
    
    # Gender
    genders = np.random.choice(
        ["male", "female", "other"],
        size=n_customers,
        p=[0.48, 0.48, 0.04]
    )
    
    customers_df = pd.DataFrame({
        "customer_id": customer_ids,
        "signup_date": signup_dates,
        "customer_segment": segments,
        "location": locations,
        "acquisition_channel": channels,
        "age": ages,
        "gender": genders
    })
    
    return customers_df

In [6]:
customers = generate_customers(N_CUSTOMERS)
customers.head()

,customer_id,signup_date,customer_segment,location,acquisition_channel,age,gender
0,C00001,2024-04-12,medium,Bhubaneswar,ads,25,male
1,C00002,2025-03-11,high,Hyderabad,referral,47,female
2,C00003,2024-09-27,low,Ahmedabad,organic,46,female
3,C00004,2024-04-16,medium,Bangalore,organic,38,male
4,C00005,2024-03-12,high,Hyderabad,referral,37,female


In [7]:
customers["customer_segment"].value_counts(normalize=True)

customer_segment
medium    0.506625
low       0.291375
high      0.202000
Name: proportion, dtype: float64

In [8]:
customers.shape

(8000, 7)

## Generate Products

We create a product catalog with:
- Multiple categories
- Category-based pricing
- Realistic cost structure

This enables:
- Revenue simulation
- Profit analysis (later of course) 😅

In [9]:
def generate_products(n_products):
    
    product_ids = [f"P{str(i).zfill(4)}" for i in range(1, n_products + 1)]
    
    categories = ["electronics", "clothing", "home", "beauty", "sports"]
    
    # Assign categories (uneven distribution for realism)
    product_categories = np.random.choice(
        categories,
        size=n_products,
        p=[0.25, 0.25, 0.2, 0.15, 0.15]
    )
    
    prices = []
    costs = []
    
    for cat in product_categories:
        
        if cat == "electronics":
            price = np.random.randint(5000, 50000)
        elif cat == "clothing":
            price = np.random.randint(500, 5000)
        elif cat == "home":
            price = np.random.randint(1000, 15000)
        elif cat == "beauty":
            price = np.random.randint(200, 3000)
        else:  # sports
            price = np.random.randint(800, 10000)
        
        # Cost = 60%–80% of price
        cost = price * np.random.uniform(0.6, 0.8)
        
        prices.append(price)
        costs.append(round(cost, 2))
    
    # Launch dates
    launch_dates = np.random.choice(
        pd.date_range(start_date - pd.Timedelta(days=180), end_date),
        size=n_products
    )
    
    products_df = pd.DataFrame({
        "product_id": product_ids,
        "category": product_categories,
        "price": prices,
        "cost": costs,
        "launch_date": launch_dates
    })
    
    return products_df

In [10]:
products = generate_products(N_PRODUCTS)
products.head()

,product_id,category,price,cost,launch_date
0,P0001,clothing,4758,3463.85,2025-12-12
1,P0002,beauty,1370,974.63,2024-11-20
2,P0003,electronics,49187,38650.16,2025-11-01
3,P0004,electronics,16775,10131.04,2025-03-03
4,P0005,home,8411,6297.29,2024-05-20


In [11]:
products["category"].value_counts()

category
clothing       34
electronics    27
beauty         21
home           21
sports         17
Name: count, dtype: int64

In [12]:
products.shape

(120, 5)

In [13]:
products[["price", "cost"]].describe()

,price,cost
count,120.000000,120.000000
mean,9812.366667,6932.811250
std,12164.712005,8751.000832
min,260.000000,180.410000
25%,2099.250000,1545.245000
50%,4234.500000,2971.070000
75%,10529.000000,7926.380000
max,49247.000000,38650.160000


## Generate Interactions

We simulate customer engagement over time:
- Behavior varies by customer segment
- Interactions are generated monthly
- Activity decays → leads to churn

This table forms the foundation for:
- Purchase behavior
- Churn signals

In [21]:
def generate_interactions(customers, products, target_interactions=250000):
    
    interaction_data = []
    interaction_id = 1
    
    product_ids = products["product_id"].values
    
    for _, customer in customers.iterrows():
        
        # Stop if target reached
        if len(interaction_data) >= target_interactions:
            break
        
        customer_id = customer["customer_id"]
        segment = customer["customer_segment"]
        signup_date = customer["signup_date"]
        
        current_date = signup_date
        last_active_date = signup_date
        
        churned = False
        
        # Segment-based configs
        if segment == "high":
            monthly_range = (20, 40)
            churn_days = 90
        elif segment == "medium":
            monthly_range = (8, 20)
            churn_days = 60
        else:
            monthly_range = (2, 8)
            churn_days = 30
        
        # Loop over months
        while current_date <= end_date and not churned:
            
            # Stop if target reached
            if len(interaction_data) >= target_interactions:
                break
            
            # Get current month's date range ONCE (performance fix)
            month_start = current_date.replace(day=1)
            month_end = month_start + pd.offsets.MonthEnd(0)
            days_in_month = pd.date_range(month_start, month_end)
            
            # Number of interactions this month
            n_interactions = np.random.randint(monthly_range[0], monthly_range[1] + 1)
            
            for _ in range(n_interactions):
                
                # Stop if target reached
                if len(interaction_data) >= target_interactions:
                    break
                
                # Sample interaction time
                interaction_time = np.random.choice(days_in_month)
                
                # Ensure interaction is AFTER signup_date
                if interaction_time < signup_date:
                    continue
                
                # Interaction type
                interaction_type = np.random.choice(
                    ["view", "click", "add_to_cart", "email_open"],
                    p=[0.5, 0.25, 0.15, 0.1]
                )
                
                # Product association
                if np.random.rand() < 0.7:
                    product_id = np.random.choice(product_ids)
                else:
                    product_id = None
                
                # Channel logic
                if interaction_type == "email_open":
                    channel = "email"
                else:
                    channel = np.random.choice(["web", "app"], p=[0.7, 0.3])
                
                # Append interaction
                interaction_data.append({
                    "interaction_id": interaction_id,
                    "customer_id": customer_id,
                    "interaction_type": interaction_type,
                    "product_id": product_id,
                    "interaction_timestamp": interaction_time,
                    "channel": channel
                })
                
                interaction_id += 1
                last_active_date = interaction_time
            
            # Move to next month
            current_date = current_date + pd.DateOffset(months=1)
            
            # Churn check
            if (current_date - last_active_date).days > churn_days:
                churned = True
    
    interactions_df = pd.DataFrame(interaction_data)
    
    return interactions_df

In [22]:
interactions = generate_interactions(customers, products)
interactions.head()

,interaction_id,customer_id,interaction_type,product_id,interaction_timestamp,channel
0,1,C00001,view,P0069,2024-04-18,app
1,2,C00001,click,None,2024-04-27,web
2,3,C00001,click,None,2024-05-06,app
3,4,C00001,click,None,2024-05-23,web
4,5,C00001,view,P0033,2024-05-05,app


In [23]:
interactions["interaction_type"].value_counts(normalize=True)

interaction_type
view           0.499224
click          0.250436
add_to_cart    0.150664
email_open     0.099676
Name: proportion, dtype: float64

In [24]:
interactions.shape

(250000, 6)

### Missing Product IDs

In [25]:
interactions["product_id"].isna().mean()

np.float64(0.29978)

### Time Check

In [26]:
interactions["interaction_timestamp"].min(), interactions["interaction_timestamp"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2026-03-31 00:00:00'))

### Sample Customer Behavior

In [27]:
sample_customer = interactions[interactions["customer_id"] == "C00001"]
sample_customer.sort_values("interaction_timestamp").head(20)

,interaction_id,customer_id,interaction_type,product_id,interaction_timestamp,channel
0,1,C00001,view,P0069,2024-04-18,app
1,2,C00001,click,None,2024-04-27,web
15,16,C00001,view,P0098,2024-05-01,app
8,9,C00001,view,P0001,2024-05-01,web
4,5,C00001,view,P0033,2024-05-05,app
2,3,C00001,click,None,2024-05-06,app
7,8,C00001,click,None,2024-05-07,web
10,11,C00001,email_open,None,2024-05-07,email
5,6,C00001,view,P0028,2024-05-08,app
16,17,C00001,click,P0010,2024-05-09,web


## Generate Transactions

Transactions are generated from interactions:
- Not all interactions lead to purchases
- Purchase probability depends on:
  - interaction type
  - customer segment

This creates realistic conversion behavior.

In [35]:
def generate_transactions(interactions, customers, products):
    
    transaction_data = []
    transaction_id = 1
    
    # Fast lookup maps
    customer_segment_map = customers.set_index("customer_id")["customer_segment"].to_dict()
    product_price_map = products.set_index("product_id")["price"].to_dict()
    
    for _, interaction in interactions.iterrows():
        
        interaction_type = interaction["interaction_type"]
        customer_id = interaction["customer_id"]
        product_id = interaction["product_id"]
        interaction_time = interaction["interaction_timestamp"]
        
        # Skip if no product associated
        if pd.isna(product_id):
            continue
        
        # Updated (balanced) conversion probabilities
        if interaction_type == "add_to_cart":
            base_prob = 0.35
        elif interaction_type == "click":
            base_prob = 0.15
        elif interaction_type == "view":
            base_prob = 0.05
        else:
            continue  # email_open doesn't convert
        
        # Segment multiplier
        segment = customer_segment_map[customer_id]
        
        if segment == "high":
            multiplier = 1.5
        elif segment == "medium":
            multiplier = 1.0
        else:
            multiplier = 0.5
        
        final_prob = base_prob * multiplier
        
        # Conversion decision
        if np.random.rand() < final_prob:
            
            quantity = np.random.randint(1, 4)
            price = product_price_map[product_id]
            
            # Discount logic
            if np.random.rand() < 0.8:
                discount = np.random.uniform(0, 0.10)
            else:
                discount = np.random.uniform(0.10, 0.30)
            
            transaction_data.append({
                "transaction_id": transaction_id,
                "customer_id": customer_id,
                "product_id": product_id,
                "transaction_date": interaction_time,
                "quantity": quantity,
                "price": price,
                "discount": round(discount, 2)
            })
            
            transaction_id += 1
    
    transactions_df = pd.DataFrame(transaction_data)
    
    return transactions_df

In [36]:
transactions = generate_transactions(interactions, customers, products)
transactions.head()

,transaction_id,customer_id,product_id,transaction_date,quantity,price,discount
0,1,C00001,P0078,2024-05-18,1,30202,0.07
1,2,C00001,P0007,2024-05-26,3,16826,0.04
2,3,C00001,P0082,2024-05-22,2,6509,0.05
3,4,C00001,P0115,2024-08-14,3,7554,0.20
4,5,C00001,P0065,2024-09-05,1,4389,0.01


In [37]:
transactions.shape

(24431, 7)

### Discount Distribution

In [38]:
transactions["discount"].describe()

count    24431.000000
mean         0.080149
std          0.070196
min          0.000000
25%          0.030000
50%          0.060000
75%          0.090000
max          0.300000
Name: discount, dtype: float64

### Quantity Check

In [39]:
transactions["quantity"].value_counts()

quantity
2    8231
3    8110
1    8090
Name: count, dtype: int64

### Revenue Sanity Check

In [40]:
(transactions["price"] * transactions["quantity"]).describe()

count     24431.000000
mean      19419.244935
std       26990.923949
min         260.000000
25%        3824.000000
50%        7915.000000
75%       22643.000000
max      147741.000000
dtype: float64

### Segment Behavior Check

In [41]:
transactions.merge(customers, on="customer_id")["customer_segment"].value_counts(normalize=True)

customer_segment
high      0.571446
medium    0.418730
low       0.009824
Name: proportion, dtype: float64

## Export Data

Saving generated datasets to `data/raw/` for downstream processing.

In [45]:
customers.to_csv("../data/raw/customers.csv", index=False)
products.to_csv("../data/raw/products.csv", index=False)
interactions.to_csv("../data/raw/interactions.csv", index=False)
transactions.to_csv("../data/raw/transactions.csv", index=False)